# Data collection

In [2]:
import cv2
import mediapipe as mp
import csv
import os
import time

label = 'I love u'  # <- CHANGE THIS for each sign
save_path = f'C:/SignlanguageAI/{label}'
os.makedirs(save_path, exist_ok=True)

mp_hands = mp.solutions.hands
hands = mp_hands.Hands(static_image_mode=False, max_num_hands=1, min_detection_confidence=0.7)
mp_drawing = mp.solutions.drawing_utils

cap = cv2.VideoCapture(0)

count = 100
total_samples = 200

while cap.isOpened() and count < total_samples:
    ret, frame = cap.read()
    if not ret:
        continue

    frame = cv2.flip(frame, 1)
    rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    result = hands.process(rgb)

    if result.multi_hand_landmarks:
        for hand_landmarks in result.multi_hand_landmarks:
            mp_drawing.draw_landmarks(frame, hand_landmarks, mp_hands.HAND_CONNECTIONS)

            # Extract landmark positions
            landmarks = []
            for lm in hand_landmarks.landmark:
                landmarks.extend([lm.x, lm.y, lm.z])

            # Save to CSV
            with open(f"{save_path}/{label}_{count}.csv", mode='w', newline='') as f:
                writer = csv.writer(f)
                writer.writerow(landmarks)

            count += 1
            print(f"Saved sample {count}/{total_samples}")

    cv2.putText(frame, f'Saving: {label} ({count}/{total_samples})', (10, 30),
                cv2.FONT_HERSHEY_SIMPLEX, 1, (255, 0, 0), 2)
    cv2.imshow('SignLang AI - Data Collector', frame)

    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()


Saved sample 101/200
Saved sample 102/200
Saved sample 103/200
Saved sample 104/200
Saved sample 105/200
Saved sample 106/200
Saved sample 107/200
Saved sample 108/200
Saved sample 109/200
Saved sample 110/200
Saved sample 111/200
Saved sample 112/200
Saved sample 113/200
Saved sample 114/200
Saved sample 115/200
Saved sample 116/200
Saved sample 117/200
Saved sample 118/200
Saved sample 119/200
Saved sample 120/200
Saved sample 121/200
Saved sample 122/200
Saved sample 123/200
Saved sample 124/200
Saved sample 125/200
Saved sample 126/200
Saved sample 127/200
Saved sample 128/200
Saved sample 129/200
Saved sample 130/200
Saved sample 131/200
Saved sample 132/200
Saved sample 133/200
Saved sample 134/200
Saved sample 135/200
Saved sample 136/200
Saved sample 137/200
Saved sample 138/200
Saved sample 139/200
Saved sample 140/200
Saved sample 141/200
Saved sample 142/200
Saved sample 143/200
Saved sample 144/200
Saved sample 145/200
Saved sample 146/200
Saved sample 147/200
Saved sample 

# prepare data

In [ ]:
import os
import pandas as pd
from sklearn.model_selection import train_test_split

data_dir = 'c:/SignlangAI/data'
all_data = []
all_labels = []

for label in os.listdir(data_dir):
    label_path = os.path.join(data_dir, label)
    for file in os.listdir(label_path):
        if file.endswith('.csv'):
            file_path = os.path.join(label_path, file)
            df = pd.read_csv(file_path, header=None)
            all_data.append(df.values.flatten())
            all_labels.append(label)

# Convert to DataFrame
X = pd.DataFrame(all_data)
y = pd.Series(all_labels)

# Save to disk (optional)
X.to_csv('c:/SignLangAI/x.csv', index=False)
y.to_csv('c:/SignLangAI/y.csv', index=False)


FileNotFoundError: [WinError 3] The system cannot find the path specified: 'c:/SignLangAI/data'

# Train model 

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.neural_network import MLPClassifier
import joblib

# Load data
X = pd.read_csv('D:/DS notes/SignLangAI/x.csv')
y = pd.read_csv('D:/DS notes/SignLangAI/y.csv')

# Encode labels
le = LabelEncoder()
y_encoded = le.fit_transform(y)

# Save label mapping
joblib.dump(le, 'label_encoder.pkl')

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(X, y_encoded, test_size=0.2, random_state=42)

# Create and train the model
model = MLPClassifier(hidden_layer_sizes=(100, 100), max_iter=500)
model.fit(X_train, y_train)

# Evaluate
accuracy = model.score(X_test, y_test)
print(f"Model accuracy: {accuracy:.2f}")

# Save model
joblib.dump(model, 'sign_classifier.pkl')


# prediction

In [ ]:
import cv2
import mediapipe as mp
import numpy as np
import joblib

# Load model and label encoder
model = joblib.load('sign_classifier.pkl')
le = joblib.load('label_encoder.pkl')

# MediaPipe setup
mp_hands = mp.solutions.hands
hands = mp_hands.Hands(min_detection_confidence=0.7, min_tracking_confidence=0.5)
mp_drawing = mp.solutions.drawing_utils

# Webcam setup
cap = cv2.VideoCapture(0)

while True:
    ret, frame = cap.read()
    if not ret:
        continue

    frame = cv2.flip(frame, 1)
    rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    result = hands.process(rgb)

    if result.multi_hand_landmarks:
        for hand_landmarks in result.multi_hand_landmarks:
            mp_drawing.draw_landmarks(frame, hand_landmarks, mp_hands.HAND_CONNECTIONS)

            # Extract landmark coordinates
            landmarks = []
            for lm in hand_landmarks.landmark:
                landmarks.extend([lm.x, lm.y, lm.z])

            # Predict if input is valid
            if len(landmarks) == 63:
                prediction = model.predict([landmarks])[0]
                sign_text = le.inverse_transform([prediction])[0]

                # Display prediction
                cv2.putText(frame, f'Sign: {sign_text}', (10, 40),
                            cv2.FONT_HERSHEY_SIMPLEX, 1.2, (0, 255, 0), 3)

    cv2.imshow('SignLang AI - Real-Time Sign Detection', frame)

    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()


# Prediction + speach

In [ ]:
import cv2
import mediapipe as mp
import numpy as np
import joblib
import pyttsx3
import time
import threading

# Load model and label encoder
model = joblib.load('sign_classifier.pkl')
le = joblib.load('label_encoder.pkl')

# MediaPipe Hands setup
mp_hands = mp.solutions.hands
hands = mp_hands.Hands(min_detection_confidence=0.7, min_tracking_confidence=0.5)
mp_drawing = mp.solutions.drawing_utils

# Timing variables
last_spoken_time = 0
speak_delay = 1  # 20 seconds

# Text-to-speech function in separate thread
def speak_text(text):
    engine = pyttsx3.init()
    engine.setProperty('rate', 150)
    engine.setProperty('volume', 1.0)
    engine.say(text)
    engine.runAndWait()
    engine.stop()

# Start webcam
cap = cv2.VideoCapture(0)

while True:
    ret, frame = cap.read()
    if not ret:
        break

    frame = cv2.flip(frame, 1)
    rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    result = hands.process(rgb)

    sign_text = ""

    if result.multi_hand_landmarks:
        for hand_landmarks in result.multi_hand_landmarks:
            mp_drawing.draw_landmarks(frame, hand_landmarks, mp_hands.HAND_CONNECTIONS)

            landmarks = []
            for lm in hand_landmarks.landmark:
                landmarks.extend([lm.x, lm.y, lm.z])

            if len(landmarks) == 63:
                current_time = time.time()
                if current_time - last_spoken_time > speak_delay:
                    prediction = model.predict([landmarks])[0]
                    sign_text = le.inverse_transform([prediction])[0]

                    print(f"Speaking: {sign_text}")
                    threading.Thread(target=speak_text, args=(sign_text,)).start()
                    last_spoken_time = current_time

    if sign_text:
        cv2.putText(frame, f'Sign: {sign_text}', (10, 40),
                    cv2.FONT_HERSHEY_SIMPLEX, 1.2, (0, 255, 0), 3)

    cv2.imshow('SignLang AI - 1s Delay Mode', frame)

    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()


Streamlit

In [ ]:
import streamlit as st
import cv2
import mediapipe as mp
import numpy as np
import joblib
import pyttsx3
import time
import threading

# ---- Helpers ----
@st.cache_resource
def load_model_and_encoder():
    model = joblib.load('sign_classifier.pkl')
    le = joblib.load('label_encoder.pkl')
    return model, le

def speak_text(text):
    engine = pyttsx3.init()
    engine.setProperty('rate', 150)
    engine.setProperty('volume', 1.0)
    engine.say(text)
    engine.runAndWait()
    engine.stop()

# ---- Load resources ----
model, le = load_model_and_encoder()

# MediaPipe Hands setup
mp_hands = mp.solutions.hands
hands = mp_hands.Hands(min_detection_confidence=0.7, min_tracking_confidence=0.5)
mp_drawing = mp.solutions.drawing_utils

# Streamlit session state initialization
if "running" not in st.session_state:
    st.session_state.running = False
if "last_spoken_time" not in st.session_state:
    st.session_state.last_spoken_time = 0.0
if "last_sign" not in st.session_state:
    st.session_state.last_sign = ""

st.title("🧠 SignLang AI – Real-Time Sign to Text + Speech")
st.markdown("Webcam feed with sign recognition and threaded TTS (1s delay).")

col1, col2 = st.columns(2)
with col1:
    if st.button("Start"):
        st.session_state.running = True
with col2:
    if st.button("Stop"):
        st.session_state.running = False

speak_delay = st.number_input("Speak delay (seconds)", min_value=0.5, max_value=10.0, value=1.0, step=0.5)
show_fps = st.checkbox("Show FPS", value=False)

video_placeholder = st.image([])
status = st.empty()

# OpenCV capture
cap = cv2.VideoCapture(0)

try:
    while st.session_state.running:
        start_time = time.time()
        ret, frame = cap.read()
        if not ret:
            status.error("Failed to grab frame from webcam.")
            break

        frame = cv2.flip(frame, 1)
        rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        result = hands.process(rgb)

        sign_text = ""

        if result.multi_hand_landmarks:
            for hand_landmarks in result.multi_hand_landmarks:
                mp_drawing.draw_landmarks(frame, hand_landmarks, mp_hands.HAND_CONNECTIONS)

                landmarks = []
                for lm in hand_landmarks.landmark:
                    landmarks.extend([lm.x, lm.y, lm.z])

                if len(landmarks) == 63:
                    current_time = time.time()
                    prediction = model.predict([landmarks])[0]
                    sign_text = le.inverse_transform([prediction])[0]

                    # TTS trigger: new sign or cooldown passed
                    if (sign_text != st.session_state.last_sign) or (current_time - st.session_state.last_spoken_time > speak_delay):
                        st.session_state.last_sign = sign_text
                        st.session_state.last_spoken_time = current_time
                        threading.Thread(target=speak_text, args=(sign_text,), daemon=True).start()

        if sign_text:
            cv2.putText(frame, f"Sign: {sign_text}", (10, 40),
                        cv2.FONT_HERSHEY_SIMPLEX, 1.2, (0, 255, 0), 3)

        if show_fps:
            fps = 1.0 / (time.time() - start_time + 1e-6)
            cv2.putText(frame, f"FPS: {fps:.1f}", (10, 80),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255, 255, 0), 2)

        video_placeholder.image(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
        status.info("Running. Show a trained sign to the camera. Press Stop to pause.")
        time.sleep(0.03)  # yield to Streamlit

    if not st.session_state.running:
        status.warning("Stopped. Click Start to resume.")

except Exception as e:
    st.error(f"Error: {e}")
finally:
    cap.release()
    cv2.destroyAllWindows()
